# Reward Model Training

In [1]:
from transformers import AutoTokenizer
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)

/home/talos-1/miniforge3/envs/torch_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# %pip install datasets==3.5.0

In [3]:
from datasets import load_dataset
dataset_name = 'sst2'
dataset = load_dataset(dataset_name)
dataset

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [4]:
ds_train, ds_val = dataset['train'], dataset['validation']

In [5]:
ds_train[4]

{'idx': 4,
 'sentence': 'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ',
 'label': 0}

## Tokenize the dataset

In [6]:
REWARD_TOKEN_ID = tokenizer.eos_token_id

In [7]:
REWARD_TOKEN_ID

50256

In [8]:
def tokenize(batch):
    outputs = tokenizer(batch['sentence'])
    outputs['score'] = [0] * len(outputs['input_ids'])
    outputs['score_index'] = [0] * len(outputs['input_ids'])
    for i in range(len(outputs['input_ids'])):
        outputs['input_ids'][i].append(REWARD_TOKEN_ID)
        outputs['attention_mask'][i].append(1)
        outputs['score'][i] = float(batch['label'][i])
        outputs['score_index'][i] = len(outputs['input_ids'][i]) - 1
    return outputs

map_kwargs = {
    "batched": True,
    "batch_size": 512,
    "remove_columns": ['idx', 'sentence', 'label']
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

Map: 100%|██████████| 872/872 [00:00<00:00, 37809.18 examples/s]


In [9]:
tokenized_dataset_train[4]

{'input_ids': [261,
  262,
  5290,
  15827,
  12,
  1659,
  12,
  1169,
  12,
  1008,
  9310,
  35478,
  20954,
  262,
  28303,
  714,
  47478,
  469,
  510,
  220,
  50256],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'score': 0.0,
 'score_index': 20}

In [10]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [11]:
tokenized_dataset_train[4]

{'input_ids': tensor([  261,   262,  5290, 15827,    12,  1659,    12,  1169,    12,  1008,
          9310, 35478, 20954,   262, 28303,   714, 47478,   469,   510,   220,
         50256]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'score': tensor(0.),
 'score_index': tensor(20)}

### Filter out shorter tweets

In [12]:
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x['input_ids']) > 6)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x['input_ids']) > 6)

Filter: 100%|██████████| 872/872 [00:00<00:00, 31345.04 examples/s]


In [13]:
len(tokenized_dataset_train)

49401

## LLM with Reward Head

In [14]:
import torch
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(self.reward.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        return self.reward(hidden_states)

class GPT2RewardHead(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_head = RewardHead(self.llm.config)

    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        last_hidden_state = transformer_outputs.hidden_states[-1]
        reward = self.reward_head(last_hidden_state).squeeze(-1)
        return torch.sigmoid(reward)


In [15]:
model = GPT2RewardHead(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2034.22it/s]


In [16]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorWithPadding(tokenizer)
dataloader_params = {
    'batch_size': 64,
    'shuffle': True,
    'collate_fn': data_collator
}
train_dataloader = DataLoader(tokenized_dataset_train, **dataloader_params)
val_dataloader = DataLoader(tokenized_dataset_val, **dataloader_params)

In [17]:
batch = next(iter(train_dataloader))
print(batch.keys())

KeysView({'input_ids': tensor([[41304,   278,   474,  ...,   764,   220, 50256],
        [  400, 11469,    82,  ..., 50256, 50256, 50256],
        [47895,   597, 17218,  ..., 50256, 50256, 50256],
        ...,
        [  293,  3080,   345,  ..., 50256, 50256, 50256],
        [  505,   286,   262,  ..., 50256, 50256, 50256],
        [  271,   523,   537,  ..., 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'score': tensor([1., 0., 0., 1., 0., 1., 0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 0.,
        0., 1., 0., 1., 1., 1., 0., 0., 1., 0., 0., 1., 1., 1., 1., 1., 0., 1.,
        0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 0.,
        1., 1., 0., 1., 0., 1., 1., 1., 1., 0.]), 'score_index': tensor([41, 16, 31,  7, 15, 10, 13, 30,  7, 11, 13,  8, 22, 15,

In [18]:
print(batch['input_ids'][1])
print(batch['attention_mask'][1])
print(batch['score'][1])
print(batch['score_index'][1])

tensor([  400, 11469,    82,   262,  5386,   656,   257,  2003,   484, 24486,
          299,   470,   881,  1337,   546,   220, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
tensor(0.)
tensor(16)


In [19]:
print(tokenizer.decode(batch['input_ids'][1]))

thrusts the audience into a future they wo n't much care about <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [20]:
batch['attention_mask'][1].nonzero()[-1]

tensor([16])

In [21]:
outputs = model(batch['input_ids'], batch['attention_mask'])

In [22]:
print(outputs.shape)

torch.Size([64, 42])


### Training Config

In [23]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()
num_epochs = 1 # N+ Implementation Detail paper


In [24]:
def validate():
    model.eval()
    total_loss = 0
    for i, batch in enumerate(val_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        with torch.no_grad():
            scores = model(**model_inputs)
            batch_indices = torch.arange(scores.shape[0])
            score = scores[batch_indices, inputs['score_index']]
            target = inputs['score']
            loss = criterion(score, target)
        total_loss += loss.item()
    print('validation loss:', total_loss / len(val_dataloader))

### Training Loop

In [25]:
model.to(device)

validate()
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, inputs['score_index']]
        target = inputs['score']
        loss = criterion(score, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(loss.item())
    validate()


validation loss: 8.042483159473964
9.559283256530762
3.1909303665161133
3.9181830883026123
3.6373367309570312
1.8578187227249146
1.1759512424468994
1.694281816482544
1.8078923225402832
1.7213037014007568
1.6860178709030151
0.8503799438476562
1.2334178686141968
0.9604930877685547
0.9379676580429077
0.8256826996803284
1.047141432762146
0.8925906419754028
0.9816186428070068
0.9119651317596436
0.8959805965423584
0.7960790991783142
0.8633370995521545
0.8342514634132385
0.8104631900787354
0.8520116209983826
0.9729285836219788
0.9102728366851807
0.8000921010971069
0.737239420413971
0.7578109502792358
0.6953839659690857
0.7206833362579346
0.8833533525466919
0.693261981010437
0.5913780927658081
0.6629618406295776
0.5767138004302979
0.7468785047531128
0.7203459739685059
0.6695045232772827
0.6306099891662598
0.6818814277648926
0.6729496717453003
0.6044243574142456
0.7183260917663574
0.6341716647148132
0.5973327159881592
0.7588659524917603
0.6551927328109741
0.6509710550308228
0.5645057559013367
0

In [26]:
torch.save(model.state_dict(), 'reward_model.pt')

In [27]:
validate()

validation loss: 0.26005631791693823


### Confusion Matrix

In [29]:
from sklearn.metrics import confusion_matrix
model.eval()

all_predictions = []
all_labels = []

for i, batch in enumerate(val_dataloader):
    inputs = batch.to(device)
    model_inputs = {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask']
    }
    with torch.no_grad():
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, inputs['score_index']]
        target = inputs['score']
    predictions = (score > 0.5).int()

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(target.cpu().numpy())

confusion_matrix(all_labels, all_predictions)

array([[381,  43],
       [ 37, 406]])